## Q. You are required to develop a suitable machine learning (ML) model to predict whether a given biological sample is cancerous or non-cancerous based on the abundance of miRNAs. The last column represents the class label (0: Normal, 1: Cancer).

## 1. Data Preprocessing

In [22]:
import pandas as pd

df = pd.read_csv("processed_label_data_uterus_transpose.csv")
print(df.shape)
df.head()

(347, 1882)


,hsa-let-7a-1,hsa-let-7a-2,hsa-let-7a-3,hsa-let-7b,hsa-let-7c,hsa-let-7d,hsa-let-7e,hsa-let-7f-1,hsa-let-7f-2,hsa-let-7g,...,hsa-mir-942,hsa-mir-943,hsa-mir-944,hsa-mir-95,hsa-mir-9500,hsa-mir-96,hsa-mir-98,hsa-mir-99a,hsa-mir-99b,Label
0,26492.18911,25465.36414,25413.83073,6936.48394,12863.78655,3790.413109,2939.849858,18591.50639,19793.54494,19851.36717,...,4.629272,0.0,0.611413,42.362208,0,0.960792,1439.616298,24739.79123,3973.749756,0
1,24500.58287,23922.61932,24024.29359,8966.13145,14068.39754,2580.177049,2398.297576,19594.53597,20787.24404,20572.47233,...,4.617875,0.0,0.243046,82.473632,0,17.499317,1642.667353,22182.65252,4159.085317,0
2,30444.97479,29176.95987,29163.05397,9442.93081,13431.89635,4850.762591,3311.812965,21561.22668,23095.11125,21691.72107,...,7.827821,0.0,0.000000,45.861822,0,3.959957,1631.133731,15497.79647,3982.150726,0
3,23039.03602,21935.53567,21919.98851,13303.38068,12468.35892,3308.876721,2038.302403,13202.67221,14083.29125,18603.33821,...,5.801179,0.0,0.116024,73.094861,0,2.204448,558.305508,18229.62623,4514.825900,0
4,34055.04786,32850.63809,32749.36640,14646.32806,13331.15238,6518.460970,3702.655823,18622.05576,20196.64973,31044.29470,...,5.063585,0.0,0.180842,33.455827,0,12.026013,1953.368183,17122.78261,4169.952345,0


In [23]:
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

In [24]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X)

selected_columns = X.columns[selector.get_support()]
X = pd.DataFrame(X_var, columns=selected_columns)

In [25]:
from scipy.stats import ttest_ind

significant_features = []

for col in X.columns:
    class0 = X[y == 0][col]
    class1 = X[y == 1][col]
    _, p = ttest_ind(class0, class1)
    
    if p < 0.05:
        significant_features.append(col)

X = X[significant_features]
print("Remaining features:", len(X.columns))

Remaining features: 652


In [26]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 2. 10-Fold Cross Validation

In [27]:
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(random_state=42)

scores = cross_val_score(model, X_scaled, y, cv=10, scoring='accuracy')

print("10-Fold Accuracy:", scores.mean())

10-Fold Accuracy: 0.985546218487395


## 3. Model Selection & Hyperparameter Optimization

In [28]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "RandomForest": RandomForestClassifier()
}

for name, model in models.items():
    scores = cross_val_score(model, X_scaled, y, cv=10)
    print(name, scores.mean())

Logistic 0.9854621848739497
SVM 0.9826890756302522
RandomForest 0.985546218487395


In [29]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=10,
    scoring='accuracy'
)

grid.fit(X_scaled, y)

print("Best Parameters:", grid.best_params_)
print("Best Score:", grid.best_score_)

Best Parameters: {'max_depth': None, 'min_samples_split': 10, 'n_estimators': 100}
Best Score: 0.9884033613445379


## 4. Feature Ranking Using Filter Methods

In [30]:
from sklearn.feature_selection import f_classif

f_values, p_values = f_classif(X_scaled, y)

feature_scores = pd.DataFrame({
    "Feature": X.columns,
    "F_score": f_values
}).sort_values(by="F_score", ascending=False)

feature_scores.head()

,Feature,F_score
651,hsa-mir-99a,310.255188
75,hsa-mir-145,262.348245
151,hsa-mir-29a,222.260485
441,hsa-mir-548aw,187.933198
4,hsa-mir-100,187.621419


In [31]:
from sklearn.feature_selection import mutual_info_classif

mi = mutual_info_classif(X_scaled, y)

mi_scores = pd.DataFrame({
    "Feature": X.columns,
    "MI_score": mi
}).sort_values(by="MI_score", ascending=False)

mi_scores.head()

,Feature,MI_score
611,hsa-mir-7-2,0.266307
612,hsa-mir-7-3,0.263177
63,hsa-mir-1307,0.261467
64,hsa-mir-130b,0.254832
58,hsa-mir-1301,0.248407


In [32]:
from sklearn.feature_selection import chi2
from sklearn.preprocessing import MinMaxScaler

X_minmax = MinMaxScaler().fit_transform(X)

chi_scores, _ = chi2(X_minmax, y)

chi_df = pd.DataFrame({
    "Feature": X.columns,
    "Chi_score": chi_scores
}).sort_values(by="Chi_score", ascending=False)

chi_df.head()

,Feature,Chi_score
651,hsa-mir-99a,38.289326
75,hsa-mir-145,29.249783
322,hsa-mir-4536-1,26.966218
323,hsa-mir-4536-2,24.997607
3,hsa-mir-1-2,24.497207


## 5. Optimal Feature Subset Selection

In [33]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split

top_features = feature_scores["Feature"].values

results = []

for k in range(5, len(top_features), 5):
    selected = top_features[:k]
    
    X_subset = X[selected]
    X_scaled = scaler.fit_transform(X_subset)
    
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=42
    )
    
    model = RandomForestClassifier(**grid.best_params_)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    results.append([
        k,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ])

results_df = pd.DataFrame(
    results,
    columns=["Num_Features","Accuracy","Precision","Recall","F1"]
)

results_df

,Num_Features,Accuracy,Precision,Recall,F1
0,5,1.000000,1.000000,1.0,1.000000
1,10,0.985714,0.983051,1.0,0.991453
2,15,1.000000,1.000000,1.0,1.000000
3,20,0.985714,0.983051,1.0,0.991453
4,25,0.985714,0.983051,1.0,0.991453
...,...,...,...,...,...
125,630,1.000000,1.000000,1.0,1.000000
126,635,1.000000,1.000000,1.0,1.000000
127,640,1.000000,1.000000,1.0,1.000000
128,645,1.000000,1.000000,1.0,1.000000
